# Variational Crack Method (VCM) — Code Organization and API Reference

**Version:** VCM_1_0  
**Date:** 2026-01-20

This notebook is a **developer-facing** description of the VCM codebase, including:

1. Auto-generated lists of **modules, classes, and key methods**
2. A concrete **solver call graph / pipeline**
3. A physics-facing map from **variational unknowns → PK force → SIF extraction**
4. A practical guide to **extending** the code (new kernels, constraints, growth laws, plots)

**Canonical update:** The solver façade `DCENetworkStaticV4` lives in  
`Usolver/parametrization.py`.



## 1. Architectural separation

VCM is structured into four layers with one-way dependencies:

```
Ugenerator  →  Usolver  →  Uprocessor  →  Uplotter
                          (optional: BEM correction)
```

- **Ugenerator**: geometry/topology generation only  
- **Usolver**: variational assembly + constraints + KKT solve  
- **Uprocessor**: kinematic and field reconstruction; PK and SIF extraction  
- **Uplotter**: visualization and diagnostics (read-only)



## 2. Terminology: Full/Half vs Dipolar/Monopolar

In this codebase, `crack_mode` refers to the **elastic source structure**, not merely geometry:

| Code term | Physical term | Interpretation |
|---|---|---|
| `full` | **dipolar** | Burgers neutrality constraints enforce a dipole-like far field |
| `half` | **monopolar** | Net Burgers content allowed; junction DOFs enforce compatibility |



## 3. Canonical solver entry point

The stable public API for solving crack networks is:

- **Module:** `Usolver/parametrization.py`
- **Class:** `DCENetworkStaticV4`
- **Method:** `solve(...)`

This façade orchestrates:
- assembly helpers in `Usolver/build.py`
- constraint builders in `Usolver/constraints.py`
- KKT solution via `solve_kkt_lsq` in `Usolver/KKT.py`



## 4. Solver pipeline and call graph

`DCENetworkStaticV4.solve()` executes the following stages:

1. Validate options; map Chebyshev aliases to `collocation_mode`
2. Compute vertex degree map: `vertex_degrees(network)`
3. Build crack polylines  
   - `build_polylines_full(network)` for `crack_mode='full'`  
   - `build_polylines_half_branches(network, deg)` for `crack_mode='half'`
4. Discretize polylines: `discretize_polylines(...)` → `poly_panels`
5. Allocate unknowns: `allocate_unknowns(...)` → `(offsets, junction_dof, nunk)`
6. Assemble operator: `assemble_operator(...)` → `(K, rhs)`
7. Build constraints  
   - `build_constraints_full(...)`  
   - `build_constraints_half_option_a(...)`
8. Solve KKT system: `solve_kkt_lsq(K, rhs, C, ridge)` → `q`
9. Package solution dictionary `sol`

### Call graph

```
DCENetworkStaticV4.solve
  ├─ vertex_degrees
  ├─ build_polylines_full / build_polylines_half_branches
  ├─ discretize_polylines
  ├─ allocate_unknowns
  ├─ assemble_operator
  ├─ build_constraints_full / build_constraints_half_option_a
  ├─ solve_kkt_lsq
  └─ sol dict
```



## 5. Physics mapping: unknowns → jump → stress → PK → SIF

### 5.1 Kinematics and displacement jump

For each polyline panel $k$, the solver returns densities $b_I^k$, $b_{II}^k$,
local tangent $\mathbf{t}_k$, normal $\mathbf{n}_k$, and panel weight $\Delta s_k$.

The incremental displacement jump is

$$
\Delta \mathbf{J}_k
= \left( b_{II}^k\, \mathbf{t}_k + b_I^k\, \mathbf{n}_k \right)\, \Delta s_k .
$$

The accumulated jump $\mathbf{J}(s)$ yields:

- **COD (opening):** $\text{COD} = \mathbf{J}\cdot\mathbf{n}$  
- **CSD (sliding):** $\text{CSD} = \mathbf{J}\cdot\mathbf{t}$



### 5.2 Stress and displacement fields

The global stress field is obtained by superposition of edge-dislocation kernels:

$$
\boldsymbol{\sigma}(\mathbf{x})
= \sum_k \boldsymbol{\sigma}^{\text{edge}}
\bigl(\mathbf{x}-\mathbf{x}_k,\; \Delta \mathbf{B}_k \bigr),
$$

where

$$
\Delta \mathbf{B}_k
= \left( b_{II}^k \mathbf{t}_k + b_I^k \mathbf{n}_k \right) \Delta s_k .
$$



### 5.3 Peach–Koehler (configurational) force

A PK-like force density is computed from stress as

$$
\mathbf{f}_{PK}
= \left( \boldsymbol{\sigma}\cdot\mathbf{b} \right)\times\mathbf{e}_z .
$$

This quantity measures configurational driving forces and is distinct from the geometric jump.



### 5.4 Stress intensity factor (SIF) extraction

Near a crack tip, COD and CSD follow square-root scaling:

$$
\text{COD}(r) \approx A_I \sqrt{\frac{r}{2\pi}}, \qquad
\text{CSD}(r) \approx A_{II} \sqrt{\frac{r}{2\pi}} .
$$

The fitted amplitudes yield

$$
K_I = \frac{\mu}{\kappa+1} A_I, \qquad
K_{II} = \frac{\mu}{\kappa+1} A_{II}.
$$



## 6. Extension guide

- **New discretization / basis:** `Usolver/build.py`  
- **New constraints / junction option:** `Usolver/constraints.py`  
- **New fracture metric:** new module under `Uprocessor`  
- **Crack growth law:** `Uprocessor/growth.py`  
- **New plot:** `Uplotter/core.py` or `Uplotter/plot_PK.py`


## 7. Notebook bootstrap (recommended)

In [1]:
import os, sys
from pathlib import Path

cwd = Path(os.getcwd()).resolve()
root = next((p for p in [cwd] + list(cwd.parents) if (p / "preamble.py").exists()), None)
if root is None:
    raise RuntimeError("preamble.py not found")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Repo root:", root)


Repo root: C:\Users\Owner\Documents\Repos\Fracture\VCM_1_0
